In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set base paths
import os
from pathlib import Path

# Drive locations (persistent across sessions)
DRIVE_ROOT = Path('/content/drive/MyDrive/shelf-oos-detection')

# Local Colab filesystem (ephemeral — wiped between sessions)
REPO_ROOT = Path('/content/shelf-oos-detection')

# Sanity check Drive is set up correctly
assert DRIVE_ROOT.exists(), f"Drive folder not found: {DRIVE_ROOT}"
print(f"Drive root: {DRIVE_ROOT}")
print(f"Repo will live at: {REPO_ROOT}")

Drive root: /content/drive/MyDrive/shelf-oos-detection
Repo will live at: /content/shelf-oos-detection


## Clone or pull the repo

In [3]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = userdata.get('GITHUB_USERNAME')
REPO_NAME = 'shelf-oos-detection'

# Build authenticated clone URL
clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'

if REPO_ROOT.exists():
    # Recurring session — just pull latest
    print("Repo already exists, pulling latest...")
    !cd {REPO_ROOT} && git pull
else:
    # First session — clone
    print("Cloning repo for the first time...")
    !git clone {clone_url} {REPO_ROOT}

# Switch working directory to the repo
os.chdir(REPO_ROOT)
print(f"\nWorking directory: {os.getcwd()}")

Cloning repo for the first time...
Cloning into '/content/shelf-oos-detection'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 4.70 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (1/1), done.

Working directory: /content/shelf-oos-detection


## Configure Git identity

In [4]:
# Use the email associated with your GitHub account
GIT_USER_NAME = "MaskedMan13"
GIT_USER_EMAIL = "prakashvini1317@example.com"

!git config --global user.name "{GIT_USER_NAME}"
!git config --global user.email "{GIT_USER_EMAIL}"

print("Git identity configured:")
!git config --global user.name
!git config --global user.email

Git identity configured:
MaskedMan13
prakashvini1317@example.com


## Symlink Data Folders

In [5]:
# Folders we want to symlink from repo → Drive
folders_to_link = ['data', 'checkpoints', 'inpainting_models', 'logs']

for folder in folders_to_link:
    repo_path = REPO_ROOT / folder
    drive_path = DRIVE_ROOT / folder

    # Make sure the Drive folder exists
    drive_path.mkdir(parents=True, exist_ok=True)

    # Remove any existing symlink/folder at the repo path
    if repo_path.is_symlink() or repo_path.exists():
        if repo_path.is_symlink():
            repo_path.unlink()
        else:
            print(f"Warning: {repo_path} exists as a real folder, skipping")
            continue

    # Create the symlink
    repo_path.symlink_to(drive_path)
    print(f"✓ {repo_path} → {drive_path}")

# Verify
print("\nVerification:")
!ls -la {REPO_ROOT} | grep -E "data|checkpoints|inpainting|logs"

✓ /content/shelf-oos-detection/data → /content/drive/MyDrive/shelf-oos-detection/data
✓ /content/shelf-oos-detection/checkpoints → /content/drive/MyDrive/shelf-oos-detection/checkpoints
✓ /content/shelf-oos-detection/inpainting_models → /content/drive/MyDrive/shelf-oos-detection/inpainting_models
✓ /content/shelf-oos-detection/logs → /content/drive/MyDrive/shelf-oos-detection/logs

Verification:
lrwxrwxrwx 1 root root   54 Apr 27 17:09 checkpoints -> /content/drive/MyDrive/shelf-oos-detection/checkpoints
lrwxrwxrwx 1 root root   47 Apr 27 17:09 data -> /content/drive/MyDrive/shelf-oos-detection/data
lrwxrwxrwx 1 root root   60 Apr 27 17:09 inpainting_models -> /content/drive/MyDrive/shelf-oos-detection/inpainting_models
lrwxrwxrwx 1 root root   47 Apr 27 17:09 logs -> /content/drive/MyDrive/shelf-oos-detection/logs


## Install Dependencies

In [6]:
# Core CV libraries
!pip install -q ultralytics albumentations opencv-python-headless

# Verify the important ones loaded
import torch
import cv2
import albumentations as A
from ultralytics import YOLO

print(f"PyTorch:       {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"OpenCV:        {cv2.__version__}")
print(f"Albumentations:{A.__version__}")
print(f"Ultralytics imported")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch:       2.10.0+cu128
CUDA available: True
OpenCV:        4.13.0
Albumentations:2.0.8
Ultralytics imported


In [7]:
!nvidia-smi

Mon Apr 27 17:10:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
with open(REPO_ROOT / '.gitignore', 'a') as f:
    f.write('\n/data\n/checkpoints\n/inpainting_models\n/logs\n')

!cd {REPO_ROOT} && git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore

no changes added to commit (use "git add" and/or "git commit -a")


In [10]:
def commit_and_push(message):
    """Commit all changes and push to GitHub. Run at end of session."""
    os.chdir(REPO_ROOT)

    # Show what's about to be committed
    print("Changes to commit:")
    !git status --short

    # Stage, commit, push
    !git add .
    !git commit -m "{message}"
    !git push

    print("\n✓ Pushed to GitHub")